In [ ]:
"""
bweight_time.ipynb

Analyze bweight trajectories over time.

Author: Stellina X. Ao
Created: 2026-07-25
Last Modified: 2026-07-25
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
)

encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

## bweight strategy scatter

In [ ]:
# plot for all sessions

In [ ]:
from core.viz import plot_trajectory

regressor = "response_left"

ax = plot_trajectory(
    x=encoder_mb.encoder_weights[
        :, encoder.reg_idxs["DMS"], encoder.dm_idxs[regressor]
    ].mean(axis=1),
    y=encoder_mf.encoder_weights[
        :, encoder.reg_idxs["DMS"], encoder.dm_idxs[regressor]
    ].mean(axis=1),
    xlabel="mb",
    ylabel="mf",
    title=rf"$\beta$ {regressor}",
)

# ax.set_xlim([-0.08, 0.02])
# ax.set_ylim([-0.08, 0.02])

In [ ]:
from core.viz import plot_trajectory

regressor = "response_prev_left"
ax = plot_trajectory(
    x=encoder_mb.encoder_weights[
        :, encoder.reg_idxs["DLS"], encoder.dm_idxs[regressor]
    ].mean(axis=1),
    y=encoder_mf.encoder_weights[
        :, encoder.reg_idxs["DLS"], encoder.dm_idxs[regressor]
    ].mean(axis=1),
    xlabel="mb",
    ylabel="mf",
    title=rf"$\beta$ {regressor}",
)

# ax.set_xlim([-0.02, 0.02])
# ax.set_ylim([-0.02, 0.0

In [ ]:
from squiggs.renderers import StrategyWeightPETHRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond

mode = "rewarded"
reg = "DLS"

r = StrategyWeightPETHRenderer(
    weights_mb=encoder_mb.encoder_weights[:, encoder.reg_idxs[reg], :],
    weights_mf=encoder_mf.encoder_weights[:, encoder.reg_idxs[reg], :],
    regressor="rewarded_corr",
    dm_idxs=encoder.dm_idxs,
    peths_mb=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    peths_mf=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=encoder.tpre,
    posts=encoder.tpost,
    binwidth_s=encoder.stepsize_s,
    tbin_centers=encoder.tbin_centers,
)

nv = NeuronViewer(num_units=encoder.num_units, render_func=r)

#### aggregate

In [ ]:
import numpy as np
from core.data import tv_vals
from core.viz import plot_trajectory, plot_2d_row
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR


def save_trajectories(encoder, encoder_mb, encoder_mf):
    for regr in encoder.tv_keys:
        norm_str = "norm" if encoder.norm else "nonorm"
        fpath = FIGURES_DIR / "time_resolved" / "bweight" / norm_str

        vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

        for val in vals:
            regressor = f"{regr}_{val}"

            try:
                weights = {
                    reg: [
                        encoder_.encoder_weights[
                            :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                        ].mean(axis=1)
                        for encoder_ in [encoder_mb, encoder_mf]
                    ]
                    for reg in encoder.regions
                }
            except KeyError:
                continue

            # mn/mx
            mn = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).min()
            )
            mx = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).max()
            )

            fig, ax = plot_2d_row(
                plot_trajectory,
                weights,
                mn=mn,
                mx=mx,
                xlabel="mb",
                ylabel="mf",
                title=rf"$\beta$ {regressor}",
            )

            save_fig(
                fig,
                fpath / "trajectory" / regressor / subj_id,
                f"{regressor}-{subj_id}_{sess_id}.png",
            )

In [ ]:
from core.data import subject_ids, session_ids

encoders = {}

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    encoders[subj_id] = {}

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(">", sess_id)
        try:
            encoder_mb = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
                strategy_filter="mb",
            )
            encoder_mb.fit_encoder()

            encoder_mf = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
                strategy_filter="mf",
            )
            encoder_mf.fit_encoder()

            encoder = make_tre(Encoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.25,
            )
            encoder.fit_encoder()

            encoders[subj_id][sess_id] = {
                "full": encoder,
                "mb": encoder_mb,
                "mf": encoder_mf,
            }

        except ValueError:
            print("blegh")
            continue

In [ ]:
from utils.viz_utils import center_title

subj_id = "MR82"
regressor = "rewarded_corr"


def plot_trajectory_sess(subj_id="MR82", regressor="rewarded_corr"):
    fig, axes = plt.subplots(
        ncols=len(encoders[subj_id]),
        nrows=2,
        figsize=(12, 4),
        sharey=True,
        sharex=True,
        tight_layout=True,
    )

    for i, encoders_sess in enumerate(encoders[subj_id].values()):
        encoder = encoders_sess["full"]
        encoder_mb = encoders_sess["mb"]
        encoder_mf = encoders_sess["mf"]

        for j, reg in enumerate(["DMS", "DLS"]):
            ax = plot_trajectory(
                x=encoder_mb.encoder_weights[
                    :, encoder.reg_idxs[reg], encoder_mb.dm_idxs[regressor]
                ].mean(axis=1),
                y=encoder_mf.encoder_weights[
                    :, encoder.reg_idxs[reg], encoder_mf.dm_idxs[regressor]
                ].mean(axis=1),
                ax=axes[j][i],
            )

            if i == 0:
                ax.set_ylabel("mf")
            if j == 1:
                ax.set_xlabel("mb")
            ax.set_title("")

    center_title(fig, axes[0], rf"$\beta$ {regressor}", fontsize=12)
    return axes

In [ ]:
plot_trajectory_sess(subj_id="MR82", regressor="rewarded_corr")
plot_trajectory_sess(subj_id="MR83", regressor="rewarded_corr")

In [ ]:
subj_id = "MR83"
num_trials = {
    subj_id: [
        encoders_sess["mb"].num_trials for encoders_sess in encoders[subj_id].values()
    ]
    for subj_id in encoders.keys()
}

fig, ax = plt.subplots()
for subj_id, num_trials_ in num_trials.items():
    ax.plot(num_trials_, label=subj_id)
fig.legend()

### scatter

In [ ]:
import numpy as np
from core.viz import plot_scatter

for regr in encoder.tv_keys:
    norm_str = "norm" if encoder.norm else "nonorm"
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / norm_str

    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        print(regressor)
        weights = {
            reg: [
                np.concatenate(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        idxs = {
            reg: np.tile(range(encoder.num_bins), encoder.psths[reg].shape[0])
            for reg in encoder.regions
        }

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            color=idxs,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(
            fig,
            fpath / "scatter" / regressor / subj_id,
            f"{regressor}-{subj_id}_{sess_id}.png",
        )

## avg bweight traces

In [ ]:
import numpy as np


def get_bw_stats(encoder):
    bw_mean = {}
    bw_std = {}
    for reg in encoder.regions:
        bw_mean[reg] = np.abs(
            encoder.encoder_weights[:, encoder.reg_idxs[reg], :]
        ).mean(axis=1)
        bw_std[reg] = np.abs(encoder.encoder_weights[:, encoder.reg_idxs[reg], :]).std(
            axis=1
        )
    return bw_mean, bw_std

In [ ]:
from core.data import tv_vals
from utils.colors import colors_strategy


def plot_bw_traces(encoders, bw_stats):
    encoder = encoders["full"]
    fig, axes = plt.subplots(
        nrows=len(encoder.tv_keys) + 2,
        ncols=len(encoder.regions),
        figsize=(6, 12),
        sharex=True,
        sharey=True,
        tight_layout=True,
    )

    for j, reg in enumerate(encoder.regions):
        i = 0
        for regr in encoder.tv_keys:
            if regr != "response_prev":
                for k, (bw_mean, bw_std) in bw_stats.items():
                    ax = axes[i][j]
                    try:
                        m = bw_mean[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                        s = bw_std[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                    except KeyError:
                        try:
                            m = bw_mean[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                            s = bw_std[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                        except KeyError:
                            continue

                    ax.plot(
                        encoders[k].tbin_centers,
                        m,
                        color=colors_strategy[k],
                        label=f"{regr}, {k}",
                    )
                    ax.fill_between(
                        encoders[k].tbin_centers,
                        m - s,
                        m + s,
                        color=colors_strategy[k],
                        alpha=0.25,
                    )
                    ax.axhline(y=0, linewidth=0.5, color="k")
                    ax.axvline(x=0, linewidth=0.5, color="k")

                    ax.set_ylim([-0.2, 0.6])
                    ax.legend(loc="upper right")
                    if i == len(encoder.tv_keys) + 2 - 1:
                        ax.set_xlabel("Trial Time (s)")
                    if j == 0:
                        ax.set_ylabel(r"$\beta$")
                    if i == 0:
                        ax.set_title(reg)
                i += 1
            else:
                for val in tv_vals[regr]:
                    for k, (bw_mean, bw_std) in bw_stats.items():
                        ax = axes[i][j]
                        try:
                            m = bw_mean[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]
                        except KeyError:
                            continue
                        s = bw_std[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]

                        ax.plot(
                            encoder.tbin_centers,
                            m,
                            color=colors_strategy[k],
                            label=f"{regr}_{val}, {k}",
                        )
                        ax.fill_between(
                            encoder.tbin_centers,
                            m - s,
                            m + s,
                            color=colors_strategy[k],
                            alpha=0.25,
                        )
                        ax.axhline(y=0, linewidth=0.5, color="k")
                        ax.axvline(x=0, linewidth=0.5, color="k")

                        ax.set_ylim([-0.2, 0.6])
                        ax.legend(loc="upper right")
                        if i == len(encoder.tv_keys) + 2 - 1:
                            ax.set_xlabel("Trial Time (s)")
                        if j == 0:
                            ax.set_ylabel(r"$\beta$")
                    i += 1
    return fig, axes

In [ ]:
encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
plot_bw_traces(encoders, bw_stats)

In [ ]:
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_a = f"{regr}_{tv_vals[regr][0]}"
        regr_b = f"{regr}_{tv_vals[regr][1]}"
        assert all(
            np.isclose(
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_a]],
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_b]],
            )
        )

### aggregate across sessions

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder
from core.data import subject_ids, session_ids
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / "averaged" / subj_id
    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(f">{sess_id}")
        encoder = make_tre(Encoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
        )

        encoder_mb = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mb",
        )

        encoder_mf = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mf",
        )

        encoder.fit_encoder()

        try:
            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()
        except ValueError:
            continue

        encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
        bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
        fig, _ = plot_bw_traces(encoders, bw_stats)

        save_fig(fig, fpath, fname=f"{sess_id}.png")